# 第 14 章　比較条件と結論

比較する期間と代表値を替え，結論のどの部分が保たれるかを調べます。

このノートの前半は，本文の説明と一緒に実行します。各場面で先に短い問いを読み，講師の案内に合わせてセルを動かし，出力の見所を確認します。後半は約20分の実習です。前半のコードは動く見本として用意してあり，作図コードをすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-daily.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

daily = pd.read_csv(csv_path, parse_dates=["dteday"])
daily["year"] = daily["dteday"].dt.year
daily["month"] = daily["dteday"].dt.month
print(f"{len(daily)} rows, {daily['dteday'].nunique()} distinct dates")
daily[["dteday", "year", "month", "cnt"]].head(3)

## 本文と一緒に実行

### 1. 年間合計と一日平均

約65%増えたという報告は何の増加でしょうか。年ごとの日数・合計・日平均を，同じ表で比べます。

In [ ]:
def year_summary(records):
    return records.groupby("year")["cnt"].agg(
        days="size",
        total="sum",
        daily_mean="mean",
        daily_median="median",
    )


annual = year_summary(daily)
annual.round(2)

In [ ]:
year_values = annual[["total", "daily_mean"]]
extra = year_values.loc[2012] - year_values.loc[2011]
percent_more = extra / year_values.loc[2011] * 100
pd.DataFrame({"difference": extra, "increase_pct": percent_more}).round(2)

**出力の見所：**合計は64.88%増，日平均は64.43%増です。2011年365日と2012年366日を，それぞれどの計算に使ったか説明してください。

### 2. どの時期の値をまとめたか

日平均付近の日ばかりだったでしょうか。日付順の図で，平均より高い日や低い日が続く場所を読みます。

In [ ]:
plt.rcParams["figure.figsize"] = (7, 3.6)
fig, ax = plt.subplots(layout="constrained")
for year, color in [(2011, "#607a88"), (2012, "#24686a")]:
    records = daily.loc[daily["year"] == year].sort_values("dteday")
    ax.plot(records["dteday"], records["cnt"], color=color, linewidth=0.9)
    ax.hlines(
        annual.loc[year, "daily_mean"], records["dteday"].min(), records["dteday"].max(),
        color="#313b45", linestyles="dashed", linewidth=1.35,
    )

sandy_day = daily.loc[daily["dteday"] == pd.Timestamp("2012-10-29")].iloc[0]
ax.scatter(sandy_day["dteday"], sandy_day["cnt"], color="#a34f30",
           edgecolors="white", s=45, zorder=3, clip_on=False)
ax.annotate("Oct 29", (sandy_day["dteday"], sandy_day["cnt"]),
            xytext=(-5, 38), textcoords="offset points", ha="right",
            arrowprops={"arrowstyle": "-", "color": "#a34f30"})
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", labelrotation=32)
ax.set_xlim(daily["dteday"].min() - pd.Timedelta(days=7), daily["dteday"].max())
ax.set(xlabel="Date", ylabel="Rentals per day", ylim=(0, 9200))
plt.show()

In [ ]:
end_of_october = daily["dteday"].between("2012-10-27", "2012-10-31")
daily.loc[end_of_october, ["dteday", "cnt"]]

**出力の見所：**10月29日の22件も，この集計版に記録された実績です。通常の営業日だけを比べたいなら，どの日を含める根拠が別に必要か考えます。

### 3. 同じ月で比べ直す

各月も一律に約65%増えたでしょうか。各年の同じ月の日平均を並べ，2011年の値を分母に増加率を求めます。

In [ ]:
month_values = daily.groupby(["month", "year"])["cnt"].mean()
monthly = month_values.unstack("year")
monthly["increase_pct"] = (monthly[2012] - monthly[2011]) / monthly[2011] * 100
monthly.loc[[3, 6, 9, 12]].round(2)

In [ ]:
plt.rcParams["figure.figsize"] = (7, 5.1)
fig, (levels, rates) = plt.subplots(2, 1, sharex=True, layout="constrained")
levels.plot(monthly.index, monthly[2011], color="#607a88", marker="o",
            linestyle="--", label="2011")
levels.plot(monthly.index, monthly[2012], color="#24686a", marker="s", label="2012")
levels.set(ylabel="Mean rentals per day", ylim=(0, 8200))
levels.legend(frameon=False, ncols=2)
rates.bar(monthly.index, monthly["increase_pct"], color="#a16142", width=0.68)
rates.axhline(percent_more["daily_mean"], color="#3c4854", linestyle="--",
             linewidth=1.2)
rates.set(xlabel="Calendar month", ylabel="Increase (%)", xticks=range(1, 13),
          xlim=(0.45, 12.55), ylim=(0, 175))
plt.show()

**出力の見所：**12か月すべてで増えましたが，増加率は約41〜157%です。増えた方向が同じであることと，大きさが同じであることを分けて報告します。

### 4. 夏と冬で必要な数を選ぶ

増加率が大きい時期と，多くの貸出に備える必要がある時期は，同じでしょうか。冬は各暦年の1・2・12月を使います。

In [ ]:
summer = daily.loc[daily["month"].isin([6, 7, 8])]
winter = daily.loc[daily["month"].isin([1, 2, 12])]

summer_summary = year_summary(summer)
winter_summary = year_summary(winter)
seasonal = pd.concat(
    [summer_summary, winter_summary],
    keys=["Jun–Aug", "Jan, Feb, Dec"],
    names=["period"],
)
seasonal.round(1)

In [ ]:
for period, summary in [("Jun–Aug", summer_summary), ("Jan, Feb, Dec", winter_summary)]:
    previous_mean = summary.loc[2011, "daily_mean"]
    increase = summary.loc[2012, "daily_mean"] - previous_mean
    increase_pct = increase / previous_mean * 100
    print(f"{period}: +{increase:,.0f} per day, +{increase_pct:.1f}%")

In [ ]:
plt.rcParams["figure.figsize"] = (6.5, 2.8)
fig, ax = plt.subplots(layout="constrained")
for row, summary in [(1, summer_summary), (0, winter_summary)]:
    first = summary.loc[2011, "daily_mean"]
    second = summary.loc[2012, "daily_mean"]
    ax.hlines(row, first, second, color="#8b98a3", linewidth=2.4)
    ax.scatter(first, row, color="#607a88", s=50, marker="o", zorder=3)
    ax.scatter(second, row, color="#24686a", s=50, marker="s", zorder=3)
    ax.annotate(f"{first:,.0f}", (first, row), xytext=(0, 9),
                textcoords="offset points", ha="center")
    ax.annotate(f"{second:,.0f}", (second, row), xytext=(0, 9),
                textcoords="offset points", ha="center")
ax.scatter([], [], color="#607a88", s=45, marker="o", label="2011")
ax.scatter([], [], color="#24686a", s=45, marker="s", label="2012")
ax.set(xlabel="Mean rentals per day", xlim=(0, 7600), ylim=(-0.45, 1.55),
       yticks=[0, 1], yticklabels=["Jan, Feb, Dec", "Jun–Aug"])
ax.legend(loc="upper left", frameon=False, ncols=2)
plt.show()

**出力の見所：**夏は約47.3%増，冬は約84.2%増です。一方，増加件数も2012年の水準も夏が大きくなります。図の線の長さと右端の位置は，何を表すでしょうか。

### 5. 真ん中の日と，基準に達した日

三日の貸出件数で，一つの低い値を替えると，平均と中央値はどう動くでしょうか。その後に，2012年の夏と冬を6,500件という教材用の目安で比べます。

In [ ]:
three_counts = pd.Series([22, 5566, 7852])
replaced_counts = pd.Series([1096, 5566, 7852])
display(pd.DataFrame({"Original": three_counts.agg(["mean", "median"]),
                      "One day replaced": replaced_counts.agg(["mean", "median"])}))
annual[["days", "daily_mean", "daily_median"]]

In [ ]:
plt.rcParams["figure.figsize"] = (6.8, 3.8)
busy_threshold = 6500
summer_2012 = summer.loc[summer["year"] == 2012, "cnt"]
winter_2012 = winter.loc[winter["year"] == 2012, "cnt"]
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, counts, title, color in [
    (axes[0], summer_2012, "Jun–Aug (92 days)", "#24686a"),
    (axes[1], winter_2012, "Jan, Feb, Dec (91 days)", "#7b8293"),
]:
    ax.hist(counts, bins=range(0, 9001, 500), color=color, edgecolor="white")
    ax.axvline(busy_threshold, linestyle="--", color="#a34f30", linewidth=1.6)
    ax.set(title=title, xlabel="Rentals per day", xlim=(0, 9000),
           xticks=[0, 3000, 6500, 9000])
axes[0].set_ylabel("Days")
plt.show()

In [ ]:
for period, counts in [("Jun–Aug", summer_2012), ("Jan, Feb, Dec", winter_2012)]:
    busy_days = (counts >= busy_threshold).sum()
    print(f"{period}: {busy_days} / {len(counts)} days")

**出力の見所：**平均・中央値は日々の分布をそれぞれ一つの数にまとめます。基準以上の日数は，元の日々の値から数えます。6,500件は実際の設備能力ではなく，この教材で比較のために置いた目安です。

## Jupyter 実習（約20分）

ここからは3〜5月か9〜11月を自分で選びます。講義の夏・冬の比較から対象期間を替え，前半の結論をそのまま使わず，選んだ期間の表で報告します。

## 1. 調べる時期を決める（約 5 分）

3〜5 月か 9〜11 月のどちらかを選びます。例えば「秋の貸出に備える担当者が，秋の一日について知りたい」のように，選ぶ理由を一文で書きます。その時期の前年比も 6〜8 月と同じくらいになりそうか，予想して理由を添えます。

次のセルには本文で使った関数が入っています。セルを実行すると，6〜8 月の日数，期間内の合計，一日平均，一日の中央値を年ごとに表示します。

（ここに選んだ時期，理由，予想を記入）

In [ ]:
def year_summary(records):
    return records.groupby("year")["cnt"].agg(
        days="size",
        total="sum",
        daily_mean="mean",
        daily_median="median",
    )


summer = daily.loc[daily["month"].isin([6, 7, 8])]
summer_summary = year_summary(summer)
summer_summary.round(2)

## 2. 選んだ月を同じ方法で集計する（約 9 分）

`months` に選んだ三つの月をリストで入れます。そのあと，`month` がリスト内にある日だけを抽出して，`chosen` に入れます。表示された日付の範囲と，各年の日数を確認してください。

続くセルには，`chosen` を本文と同じ関数へ渡し，表示した表を作る処理が入っています。2011 年・2012 年それぞれの平均を読むときは，何日分がその数に入ったかを一緒に見ます。平均と中央値の増加率は，それぞれ 2011 年の値を分母に計算しています。

In [ ]:
months = None
# ここに，選んだ三つの月を months に入れる一行を書く。

chosen = None
if months is not None:
    # ここに，月の条件で日を選び chosen に入れる一行を書く。
    pass

if chosen is None:
    print("月のリストと抽出の両方を書くと，選んだ日数が出ます。")
else:
    check = chosen.groupby("year")["dteday"].agg(days="size", first="min", last="max")
    display(check)

In [ ]:
if chosen is None:
    print("日付の選択まで書いてから，このセルを実行します。")
else:
    chosen_summary = year_summary(chosen)
    comparison = pd.concat(
        [summer_summary, chosen_summary],
        keys=["Jun–Aug", str(months)],
        names=["period"],
    )
    display(comparison[["days", "total", "daily_mean", "daily_median"]].round(2))

    changes = []
    for period, summary in [("Jun–Aug", summer_summary), (str(months), chosen_summary)]:
        daily_values = summary[["daily_mean", "daily_median"]]
        first_year = daily_values.loc[2011]
        difference = daily_values.loc[2012] - first_year
        growth = difference / first_year * 100
        changes.append(growth.rename(period))

    change_table = pd.DataFrame(changes)
    change_table.columns = ["Mean increase (%)", "Median increase (%)"]
    display(change_table.round(2))

<details>
<summary>月の入力と選択の書き方</summary>

3〜5 月を選んだ場合は，`months = None` の下に次の一行を書きます。9〜11 月を選んだ場合はリストの三つの値を変えます。

```python
months = [3, 4, 5]
```

`if months is not None:` の中で，`month` がそのリストにある日を選びます。

```python
    chosen = daily.loc[daily["month"].isin(months)]
```

</details>

## 3. どこまで同じ説明ができるか（約 6 分）

選んだ時期の貸出体制を考える担当者に向けて，次の三点が伝わるよう 3〜4 文で報告します。

1. 比べた時期と両年の日数は何か。一日平均と中央値はそれぞれ何件から何件になったか。
2. 6〜8 月と比べて，増加の向きは同じか。平均の増加率と，2012 年の件数そのものを読んだとき，計画に伝えたいことは何か。
3. 選んだ時期の件数を翌年にも当てはめたいなら，何を追加で調べるか。

（ここに報告を記入）

カーネルを再起動し，上から全セルを実行します。比較表と文章を残したノートブックを提出します。

### 一日の目安を替える（任意）

選んだ時期の 2012 年に限って，6,500 件以上の日と 7,000 件以上の日をそれぞれ数えます。二つとも練習用の仮の目安です。数える日付の範囲を揃えたまま，目安を高くすると何日減るか，その結果が平均の比較からは見えなかった理由も考えます。

### 出典

Capital Bikeshare の日別集計は，Hadi Fanaee-T が UCI Machine Learning Repository で公開した `day.csv` を教材用に改名したものです。 [データ配布元](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)